In [0]:
%run ../../../utils/pipeline_utils

In [0]:
# Databricks notebook source
# =============================================================================
# common/prime_dim_award.py
# Primes assist_catalog.common.dim_award
#
# Source tables (Silver):
#   silver_aasbs_award             → base award record
#   silver_aasbs_award_mod         → latest mod details (status, dates, mod_num)
#   silver_aasbs_lu_contract_type  → contract_type_desc decoded
#   silver_aasbs_lu_vehicle_type   → vehicle_type_desc decoded
#
# Grain       : One row per award.id, joined to its latest_award_mod_id
# SCD Type    : 2 — eff_start_dt=NOW, eff_end_dt=NULL, is_current_flag=TRUE
# Idempotent  : YES — TRUNCATE then INSERT
# Dependencies: dim_ia (for ia_sk), dim_agency (for agency_sk)
#
# Source column notes:
#   award.id                     → award_id (natural key)
#   award.award_piid             → award_piid
#   award.award_status_cd        → award_status_cd
#   award.latest_award_mod_id    → joins to award_mod.id for latest mod details
#   award.award_fin              → fin_code (Financial Identification Number)
#   award.acquisition_id         → links to acquisition, not needed on dim_award
#   award_mod.id                 → award_mod_id
#   award_mod.mod_num            → award_mod_num
#   award_mod.m1p_mod_start_dt   → award_start_dt (base award date on mod 0)
#   award_mod.co_signature_dt    → last_mod_dt (when CO signed latest mod)
#   award_mod.budget_fy          → not stored on dim_award; available on line_item
#   award.lead_svc_activity_address_cd → AAC used to resolve agency_sk
# =============================================================================

# COMMAND ----------
# MAGIC %run ../utils/pipeline_utils
# COMMAND ----------
dbutils.widgets.text("run_id",   "", "Pipeline Run ID")
dbutils.widgets.text("job_name", "dp1_prime_full", "Job Name")

RUN_ID   = dbutils.widgets.get("run_id")   or "manual-" + get_spark_app_id()
JOB_NAME = dbutils.widgets.get("job_name")

TARGET        = gold("common", "dim_award")
TASK          = "prime_dim_award"

S_AWARD       = silver("aasbs", "award")
S_MOD         = silver("aasbs", "award_mod")
S_CT          = silver("aasbs", "lu_contract_type")
S_VT          = silver("aasbs", "lu_vehicle_type")
G_IA          = gold("common", "dim_ia")
G_AGENCY      = gold("common", "dim_agency")

print(f"[{TASK}] target={TARGET}")
print(S_AWARD)
# COMMAND ----------
start_ts = audit_start(spark, RUN_ID, JOB_NAME, TASK, TARGET,
                        source_schema="aasbs", source_table="award")

# COMMAND ----------
try:
    truncate_gold(spark, TARGET)

    # award_sk is GENERATED ALWAYS AS IDENTITY — excluded from INSERT col list.
    #
    # Mod count: derived by counting all non-deleted award_mod rows per award_id.
    # award_start_dt: base award date = m1p_mod_start_dt on the first mod (mod_num = '0'
    #   or the earliest created_dt). Fallback to award.created_dt.
    # award_end_dt: from latest mod's m1p_mod_start_dt + PoP, not directly stored;
    #   we use the latest mod's co_signature_dt as a proxy for last_mod_dt.
    #   True PoP end date comes from line_item.line_item_end_dt (populated on fact).
    # vehicle_type_cd: not on award or award_mod directly in source DDL;
    #   this is an ASSIST-system concept. Set NULL on prime, enriched via delta refresh
    #   from contract vehicle data (IDV relationship or acquisition type).

    spark.sql(f"""
        INSERT INTO {TARGET}
        (
            award_id, award_piid, award_mod_num, award_mod_id,
            fin_code,
            award_status_cd, award_status_desc,
            contract_type_cd, contract_type_desc,
            vehicle_type_cd, vehicle_type_desc,
            ia_sk, agency_sk,
            award_start_dt, award_end_dt, base_award_dt, last_mod_dt,
            total_mods_count,
            eff_start_dt, eff_end_dt, is_current_flag,
            _gold_created_at, _gold_updated_at, _source_batch_id
        )
        WITH mod_counts AS (
            -- Pre-aggregate mod counts per award to avoid fan-out in the main join
            SELECT
                award_id,
                COUNT(*)                        AS total_mods_count,
                MIN(CASE WHEN mod_num = '00000' OR mod_num = ''
                         THEN m1p_mod_start_dt END) AS base_award_dt
            FROM {S_MOD}
            WHERE NVL(is_deleted,FALSE) = FALSE
            GROUP BY award_id
        ),
        award_with_mod AS (
            SELECT
                a.id                            AS award_id,
                a.award_piid,
                a.award_status_cd,
                a.award_fin                     AS fin_code,
                a.lead_svc_activity_address_cd  AS aac,
                -- Latest mod details
                m.id                            AS award_mod_id,
                m.mod_num                       AS award_mod_num,
                m.m1p_mod_start_dt              AS award_start_dt,
                -- award_end_dt not directly on mod; NULL on prime
                CAST(NULL AS TIMESTAMP)         AS award_end_dt,
                m.co_signature_dt               AS last_mod_dt,
                -- Contract type: stored on award_mod as award_form_cd in some versions;
                -- ASSIST stores contract type at the line_item level.
                -- Use lu_contract_type via line_item in the fact notebook.
                -- Set NULL here; enriched via dim_line_item in delta refresh.
                CAST(NULL AS STRING)            AS contract_type_cd,
                mc.total_mods_count,
                mc.base_award_dt
            FROM {S_AWARD} a
            -- Join to latest mod
            LEFT JOIN {S_MOD} m
                ON m.id = a.latest_award_mod_id
                AND NVL(m.is_deleted,FALSE) = FALSE
            LEFT JOIN mod_counts mc
                ON mc.award_id = a.id
            WHERE NVL(a.is_deleted,FALSE) = FALSE
        ),
        with_decoded AS (
            SELECT
                aw.*,
                COALESCE(ct.description, aw.contract_type_cd) AS contract_type_desc,
                -- vehicle_type not in source DDL columns; NULL on prime
                CAST(NULL AS STRING)                           AS vehicle_type_cd,
                CAST(NULL AS STRING)                           AS vehicle_type_desc
            FROM award_with_mod aw
            LEFT JOIN {S_CT} ct
                ON aw.contract_type_cd = ct.cd
                AND NVL(ct.is_deleted,FALSE) = FALSE
        ),
        with_fks AS (
            SELECT
                wd.*,
                -- ia_sk: award links to ia via acquisition.ia_id, but acquisition
                -- is not needed for this dim. Resolution via award_mod's award_id
                -- traced back through loa_ledger.ia_id on delta refresh.
                -- On prime, set NULL and resolve in fact build via loa_ledger.ia_id.
                CAST(NULL AS BIGINT)            AS ia_sk,
                ag.agency_sk
            FROM with_decoded wd
            LEFT JOIN {G_AGENCY} ag
                ON wd.aac = ag.activity_address_cd
                AND NVL(ag.is_current_flag,FALSE) = TRUE
        )
        SELECT
            award_id,
            award_piid,
            award_mod_num,
            award_mod_id,
            fin_code,
            award_status_cd,
            -- award_status_desc: decoded from lu_award_mod_status (DP11 ref).
            -- Not joined here to keep this notebook self-contained.
            -- Use award_status_cd directly; delta refresh enriches with desc.
            COALESCE(award_status_cd, 'UNKNOWN')    AS award_status_desc,
            contract_type_cd,
            contract_type_desc,
            vehicle_type_cd,
            vehicle_type_desc,
            ia_sk,
            agency_sk,
            award_start_dt,
            award_end_dt,
            base_award_dt,
            last_mod_dt,
            COALESCE(total_mods_count, 0)            AS total_mods_count,
            -- SCD2 initial values
            CURRENT_TIMESTAMP()                     AS eff_start_dt,
            CAST(NULL AS TIMESTAMP)                 AS eff_end_dt,
            TRUE                                    AS is_current_flag,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), '{RUN_ID}'
        FROM with_fks
    """)

    n = row_count(spark, TARGET)
    no_piid = spark.sql(
        f"SELECT COUNT(*) AS n FROM {TARGET} WHERE award_piid IS NULL"
    ).collect()[0]["n"]
    print(f"  [OK] Inserted {n:,} award rows ({no_piid} with NULL PIID)")

    audit_success(spark, RUN_ID, TARGET, n, n, start_ts)

except Exception as e:
    audit_fail(spark, RUN_ID, TARGET, str(e), traceback.format_exc(), start_ts)
    raise

# COMMAND ----------
#dbutils.notebook.exit("SUCCESS")
